In [1]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

# -------------------------------------------------
# Inputs
# -------------------------------------------------

params = {
    "x": 587288,
    "y": 1163895,
    "fault_az": 50,        # degrees clockwise from north
    "swath_width": 500,      # m
    "swath_length": 15000,   # m
}


crs = "EPSG:32619"
outfile = "/Volumes/Venezuela/Venezuela2026Doublet/GIS/MoronSwath.geojson"

# -------------------------------------------------
# Swath orientation (perpendicular to fault)
# -------------------------------------------------

profile_az = (params["fault_az"] + 90.0) % 360

theta = np.deg2rad(profile_az)

# Unit vector along swath
ux = np.sin(theta)
uy = np.cos(theta)

# Unit vector to the left of swath
vx = -uy
vy = ux

L = params["swath_length"] / 2.0
W = params["swath_width"] / 2.0

x = params["x"]
y = params["y"]

# -------------------------------------------------
# Rectangle corners
# -------------------------------------------------

corners = np.array([
    [x - L*ux - W*vx, y - L*uy - W*vy],
    [x + L*ux - W*vx, y + L*uy - W*vy],
    [x + L*ux + W*vx, y + L*uy + W*vy],
    [x - L*ux + W*vx, y - L*uy + W*vy],
])

poly = Polygon(corners)

# -------------------------------------------------
# Save GeoJSON
# -------------------------------------------------

gdf = gpd.GeoDataFrame(
    {
        "fault_az": [params["fault_az"]],
        "profile_az": [profile_az],
        "length_m": [params["swath_length"]],
        "width_m": [params["swath_width"]],
    },
    geometry=[poly],
    crs=crs,
)

gdf.to_file(outfile, driver="GeoJSON")

print(f"Saved {outfile}")

Saved /Volumes/Venezuela/Venezuela2026Doublet/GIS/MoronSwath.geojson


## xy Text file to geojson points

In [2]:
import pandas as pd
import geopandas as gpd

# Read the tab-delimited file
defo = pd.read_csv(
    "/Volumes/Venezuela/Venezuela2026Doublet/GIS/surface_deformation.disp",
    sep="\t"
)

# Remove whitespace from column names
defo.columns = defo.columns.str.strip()

# Convert longitudes from 0–360 to -180–180
defo["Longitude"] = ((defo["Longitude"] + 180) % 360) - 180

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(
    defo,
    geometry=gpd.points_from_xy(
        defo["Longitude"],
        defo["Latitude"]
    ),
    crs="EPSG:4326"
)

# Save
gdf.to_file(
    "/Volumes/Venezuela/Venezuela2026Doublet/GIS/surface_deformation.geojson",
    driver="GeoJSON"
)